# Tomahawk Cassette Rev3.2 — Broad Full-Incident Area-Corrected Validation

This notebook preserves the original 10,935-row, 4 MW/m² broad challenge and all
Rev3.1 gates. It corrects the convective thermal resistance by using the declared
embedded-channel wetted perimeter and length:

\[
\Delta T_{conv}=\frac{Q}{hA_{eff}}
=\frac{q''_{face}}{h(A_{eff}/A_{face})}.
\]

The declared 85% effective-area utilization is a proposed cassette design
requirement—not measured evidence. The notebook also reports sensitivity from 65%
to 100%. Its boiling test remains a saturation-margin screen, not CHF validation.

In [1]:
import hashlib
import json
import math
import platform
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260827
np.random.seed(SEED)

EXPECTED_BRANCH = "full_incident_broad_challenge_rev32_area_corrected"
PREFERRED_INPUT_NAME = "Tomahawk_Cassette_Rev3_2_Broad_Full_Incident_Area_Corrected_Input.csv"
EXPECTED_INPUT_SHA256 = "027afd5d2cc160bfc769c6b3ba3ab45a611fc5163b90e21016bd5a88c55e14e5"
RUN_SLUG = "broad_full_incident_area_corrected"
OUT_DIR = Path("/kaggle/working/tomahawk_rev32_area_corrected") if Path("/kaggle/working").exists() else Path.cwd() / "tomahawk_rev32_area_corrected_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONSTANTS = {
    "n_cassettes_per_module": 4,
    "cassette_area_m2": 2.5,
    "rf_power_absorbed_MW_module": 15.0,
    "rho_kg_m3": 950.0,
    "mu_Pa_s": 2.30e-4,
    "water_k_W_mK": 0.68,
    "cp_J_kgK": 4180.0,
    "cucrzr_k_W_mK": 300.0,
    "minor_loss_K": 3.0,
    "pump_efficiency": 0.70,
}

GATES = {
    "max_CuCrZr_temperature_C": 300.0,
    "minimum_boiling_margin_C": 10.0,
    "max_velocity_m_s": 5.0,
    "max_pressure_drop_MPa": 0.25,
    "max_pump_power_MW_module": 1.0,
    "max_flow_area_fraction": 0.08,
}

TOPOLOGY = {
    "parallel_smooth": {"Nu_multiplier": 1.00, "dp_multiplier": 1.00},
    "parallel_ribbed": {"Nu_multiplier": 1.25, "dp_multiplier": 1.60},
}

REQUIRED = [
    "analysis_branch", "topology", "q_nuclear_areal_MW_m2", "rf_loss_fraction",
    "pressure_MPa", "inlet_temp_C", "deltaT_C", "channel_count_per_cassette",
    "Dh_mm", "cucrzr_skin_mm", "channel_length_m", "coolant_decks",
    "channel_shape", "heated_perimeter_factor", "effective_wetted_area_efficiency",
]


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def discover_input():
    paths = list(Path.cwd().glob("*.csv"))
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        paths.extend(kaggle_root.rglob("*.csv"))
    paths = sorted({p.resolve() for p in paths if p.is_file()}, key=str)
    matches = []
    rejected = []
    for path in paths:
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            rejected.append((str(path), f"unreadable: {type(exc).__name__}"))
            continue
        missing = [c for c in REQUIRED if c not in frame.columns]
        if missing:
            rejected.append((str(path), "schema mismatch"))
            continue
        branches = set(frame["analysis_branch"].dropna().astype(str).unique())
        if branches != {EXPECTED_BRANCH}:
            rejected.append((str(path), f"branch mismatch: {sorted(branches)}"))
            continue
        matches.append((path, frame))
    preferred = [x for x in matches if x[0].name.casefold() == PREFERRED_INPUT_NAME.casefold()]
    pool = preferred or matches
    if not pool:
        inventory = "\n".join(f"  - {p}: {why}" for p, why in rejected)
        raise FileNotFoundError(f"No CSV matched {EXPECTED_BRANCH!r}.\n{inventory}")
    hashes = {sha256(p) for p, _ in pool}
    if len(pool) > 1 and len(hashes) > 1:
        raise RuntimeError("Multiple nonidentical Rev3.2 inputs were found; attach only one.")
    return pool[0]


def validate_input(path, frame):
    if frame.empty:
        raise ValueError("Input has zero rows.")
    if len(frame) != 10_935:
        raise ValueError(f"Expected the unchanged 10,935-row challenge grid, found {len(frame):,}.")
    numeric = [c for c in REQUIRED if c not in {"analysis_branch", "topology", "channel_shape"}]
    if not np.isfinite(frame[numeric].to_numpy(float)).all():
        raise ValueError("Non-finite numeric input found.")
    if not set(frame["topology"]).issubset(TOPOLOGY):
        raise ValueError("Unknown topology found.")
    if set(frame["channel_shape"].astype(str)) != {"square_embedded"}:
        raise ValueError("Rev3.2 requires the declared square_embedded channel shape.")
    if not (frame["coolant_decks"] == 1).all():
        raise ValueError("This frozen Rev3.2 branch requires one coolant deck.")
    if not np.allclose(frame["heated_perimeter_factor"], 4.0):
        raise ValueError("Expected four heated sides for the embedded square-channel model.")
    if not np.allclose(frame["effective_wetted_area_efficiency"], 0.85):
        raise ValueError("Expected the declared 0.85 effective-area design requirement.")
    actual_hash = sha256(path)
    if path.name.casefold() == PREFERRED_INPUT_NAME.casefold() and actual_hash != EXPECTED_INPUT_SHA256:
        raise ValueError(f"Input hash mismatch: {actual_hash}")
    return actual_hash


def saturation_temperature_C(p_MPa):
    return np.interp(np.asarray(p_MPa, dtype=float), [3, 4, 5, 6, 7], [233.9, 250.4, 263.9, 275.6, 285.8])


def evaluate(frame, efficiency_override=None):
    x = frame.copy()
    c = CONSTANTS
    total_area = c["n_cassettes_per_module"] * c["cassette_area_m2"]
    q_nuclear_W = x["q_nuclear_areal_MW_m2"].to_numpy(float) * 1e6 * total_area
    q_rf_W = c["rf_power_absorbed_MW_module"] * 1e6 * x["rf_loss_fraction"].to_numpy(float)
    q_total_W = q_nuclear_W + q_rf_W
    dT = x["deltaT_C"].to_numpy(float)
    mdot_module = q_total_W / (c["cp_J_kgK"] * dT)
    mdot_cassette = mdot_module / c["n_cassettes_per_module"]

    Dh = x["Dh_mm"].to_numpy(float) / 1000.0
    n_channels = x["channel_count_per_cassette"].to_numpy(float)
    decks = x["coolant_decks"].to_numpy(float)
    flow_area = decks * n_channels * Dh**2
    velocity = mdot_cassette / (c["rho_kg_m3"] * flow_area)
    Re = c["rho_kg_m3"] * velocity * Dh / c["mu_Pa_s"]
    Pr = c["cp_J_kgK"] * c["mu_Pa_s"] / c["water_k_W_mK"]
    Nu_base = np.where(Re < 2300.0, 4.36, 0.023 * Re**0.8 * Pr**0.4)
    nu_mult = x["topology"].map({k: v["Nu_multiplier"] for k, v in TOPOLOGY.items()}).to_numpy(float)
    dp_mult = x["topology"].map({k: v["dp_multiplier"] for k, v in TOPOLOGY.items()}).to_numpy(float)
    Nu = Nu_base * nu_mult
    h = Nu * c["water_k_W_mK"] / Dh

    friction_base = np.where(Re < 2300.0, 64.0 / np.maximum(Re, 1.0), 0.3164 / Re**0.25)
    dynamic_pressure = 0.5 * c["rho_kg_m3"] * velocity**2
    length = x["channel_length_m"].to_numpy(float)
    # Rib roughness multiplies distributed friction, not the independent header K term.
    dp_Pa = (friction_base * dp_mult * length / Dh + c["minor_loss_K"]) * dynamic_pressure

    perimeter_factor = x["heated_perimeter_factor"].to_numpy(float)
    wetted_area = decks * n_channels * perimeter_factor * Dh * length
    if efficiency_override is None:
        area_efficiency = x["effective_wetted_area_efficiency"].to_numpy(float)
    else:
        area_efficiency = np.full(len(x), float(efficiency_override))
    effective_wetted_area = area_efficiency * wetted_area
    effective_area_ratio = effective_wetted_area / c["cassette_area_m2"]
    if np.any(effective_area_ratio <= 0.0):
        raise ValueError("Nonpositive effective heat-transfer area ratio.")

    q_face_W_m2 = q_total_W / total_area
    skin = x["cucrzr_skin_mm"].to_numpy(float) / 1000.0
    # Q/(h*A_eff) = q_face/(h*(A_eff/A_face)). Rev3.1 omitted this area ratio.
    convective_rise_C = q_face_W_m2 / (h * effective_area_ratio)
    conduction_rise_C = q_face_W_m2 * skin / c["cucrzr_k_W_mK"]
    wall_temp = x["inlet_temp_C"].to_numpy(float) + dT + convective_rise_C + conduction_rise_C
    Tsat = saturation_temperature_C(x["pressure_MPa"])
    pump_power_W = dp_Pa * (mdot_module / c["rho_kg_m3"]) / c["pump_efficiency"]
    flow_area_fraction = flow_area / c["cassette_area_m2"]

    x["q_total_MW_module"] = q_total_W / 1e6
    x["mdot_kg_s_module"] = mdot_module
    x["flow_area_m2_cassette"] = flow_area
    x["flow_area_fraction"] = flow_area_fraction
    x["velocity_m_s"] = velocity
    x["Re"] = Re
    x["Nu"] = Nu
    x["h_W_m2K"] = h
    x["wetted_area_m2_cassette"] = wetted_area
    x["effective_wetted_area_m2_cassette"] = effective_wetted_area
    x["effective_area_ratio"] = effective_area_ratio
    x["convective_rise_C"] = convective_rise_C
    x["conduction_rise_C"] = conduction_rise_C
    x["pressure_drop_kPa"] = dp_Pa / 1000.0
    x["pump_power_MW_module"] = pump_power_W / 1e6
    x["wall_temp_C"] = wall_temp
    x["saturation_temp_C"] = Tsat
    x["boiling_margin_C"] = Tsat - wall_temp
    x["thermal_margin_C"] = GATES["max_CuCrZr_temperature_C"] - wall_temp
    x["gate_temperature"] = wall_temp <= GATES["max_CuCrZr_temperature_C"]
    x["gate_boiling_margin"] = (Tsat - wall_temp) >= GATES["minimum_boiling_margin_C"]
    x["gate_velocity"] = velocity <= GATES["max_velocity_m_s"]
    x["gate_pressure_drop"] = dp_Pa <= GATES["max_pressure_drop_MPa"] * 1e6
    x["gate_pump_power"] = pump_power_W <= GATES["max_pump_power_MW_module"] * 1e6
    x["gate_flow_area"] = flow_area_fraction <= GATES["max_flow_area_fraction"]
    gate_cols = [column for column in x if column.startswith("gate_")]
    x["passes_all_gates"] = x[gate_cols].all(axis=1)
    return x


def failure_matrix(results):
    names = {
        "gate_temperature": "Temperature",
        "gate_boiling_margin": "Boiling margin",
        "gate_velocity": "Velocity",
        "gate_pressure_drop": "Pressure drop",
        "gate_pump_power": "Pump power",
        "gate_flow_area": "Flow area",
    }
    gate_cols = list(names)
    failed = results.loc[~results["passes_all_gates"], gate_cols].copy()
    failed["failure_group"] = failed.apply(
        lambda row: " + ".join(names[g] for g in gate_cols if not bool(row[g])), axis=1
    )
    return failed.groupby("failure_group").size().sort_values(ascending=False).rename("rows").reset_index()


def main():
    input_path, inputs = discover_input()
    input_hash = validate_input(input_path, inputs)
    print("Python:", platform.python_version())
    print("Selected input:", input_path)
    print("Rows:", len(inputs))
    print("Input SHA-256:", input_hash)
    print("Input validation: PASS")

    start = time.perf_counter()
    results = evaluate(inputs)
    elapsed = time.perf_counter() - start
    gate_cols = [column for column in results if column.startswith("gate_")]
    pass_count = int(results["passes_all_gates"].sum())
    pass_percent = float(100.0 * results["passes_all_gates"].mean())
    acceptance_passed = pass_percent >= 60.0

    gate_summary = pd.DataFrame({
        "gate": gate_cols,
        "passing_rows": [int(results[g].sum()) for g in gate_cols],
        "pass_percent": [float(100.0 * results[g].mean()) for g in gate_cols],
    })
    failures = failure_matrix(results)

    sensitivity_rows = []
    for efficiency in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 1.00]:
        trial = evaluate(inputs, efficiency_override=efficiency)
        sensitivity_rows.append({
            "effective_wetted_area_efficiency": efficiency,
            "passing_rows": int(trial["passes_all_gates"].sum()),
            "pass_percent": float(100.0 * trial["passes_all_gates"].mean()),
        })
    sensitivity = pd.DataFrame(sensitivity_rows)

    feasible = results[results["passes_all_gates"]].copy()
    feasible["ranking_score"] = (
        feasible["thermal_margin_C"]
        + 0.20 * feasible["boiling_margin_C"]
        - 0.08 * feasible["pressure_drop_kPa"]
        - 2.0 * np.maximum(feasible["velocity_m_s"] - 3.0, 0.0)
    )
    top = feasible.sort_values("ranking_score", ascending=False).head(100)

    summary = pd.DataFrame([{
        "analysis_branch": EXPECTED_BRANCH,
        "rows": len(results),
        "legacy_rev31_passing_rows": 672,
        "legacy_rev31_pass_percent": 100.0 * 672 / 10_935,
        "rev32_passing_rows": pass_count,
        "rev32_pass_percent": pass_percent,
        "acceptance_rule": "pass_percent >= 60.0",
        "acceptance_passed": acceptance_passed,
        "elapsed_seconds": elapsed,
        "minimum_passing_mass_flow_kg_s_module": float(feasible["mdot_kg_s_module"].min()) if len(feasible) else np.nan,
        "maximum_passing_mass_flow_kg_s_module": float(feasible["mdot_kg_s_module"].max()) if len(feasible) else np.nan,
    }])

    prefix = f"Tomahawk_Cassette_Rev3_2_{RUN_SLUG}"
    paths = {
        "results": OUT_DIR / f"{prefix}_All_Results.csv",
        "summary": OUT_DIR / f"{prefix}_Summary.csv",
        "gates": OUT_DIR / f"{prefix}_Gate_Rates.csv",
        "failures": OUT_DIR / f"{prefix}_Failure_Matrix.csv",
        "sensitivity": OUT_DIR / f"{prefix}_Area_Efficiency_Sensitivity.csv",
        "top": OUT_DIR / f"{prefix}_Top_Candidates.csv",
        "plot": OUT_DIR / f"{prefix}_Results.png",
        "manifest": OUT_DIR / f"{prefix}_Manifest.json",
    }
    results.to_csv(paths["results"], index=False)
    summary.to_csv(paths["summary"], index=False)
    gate_summary.to_csv(paths["gates"], index=False)
    failures.to_csv(paths["failures"], index=False)
    sensitivity.to_csv(paths["sensitivity"], index=False)
    top.to_csv(paths["top"], index=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].plot(100 * sensitivity["effective_wetted_area_efficiency"], sensitivity["pass_percent"], marker="o")
    axes[0].axhline(60, color="black", linestyle="--", linewidth=1)
    axes[0].set(xlabel="Effective wetted-area utilization [%]", ylabel="Simultaneous pass rate [%]", title="Design requirement sensitivity")
    axes[0].grid(alpha=0.25)
    axes[1].bar(gate_summary["gate"].str.replace("gate_", "", regex=False), gate_summary["pass_percent"], color="#2E74B5")
    axes[1].set(ylabel="Gate pass rate [%]", title="Rev3.2 individual gates", ylim=(0, 105))
    axes[1].tick_params(axis="x", rotation=35)
    colors = np.where(results["passes_all_gates"], "#2E7D32", "#B71C1C")
    axes[2].scatter(results["velocity_m_s"], results["wall_temp_C"], c=colors, s=7, alpha=0.25)
    axes[2].axvline(GATES["max_velocity_m_s"], color="black", linestyle="--", linewidth=1)
    axes[2].axhline(GATES["max_CuCrZr_temperature_C"], color="black", linestyle="--", linewidth=1)
    axes[2].set(xlabel="Velocity [m/s]", ylabel="Wall temperature [°C]", title="Remaining failure map")
    fig.tight_layout()
    fig.savefig(paths["plot"], dpi=180)
    plt.close(fig)

    manifest = {
        "model_revision": "Rev3.2",
        "input_file": str(input_path),
        "input_sha256": input_hash,
        "rows": len(results),
        "equation_change": "Explicit Q/(h*A_effective) using embedded-channel wetted perimeter, length, and declared utilization efficiency.",
        "pressure_drop_change": "Rib multiplier applied to distributed friction only; independent header minor-loss K retained separately.",
        "gates_unchanged_from_rev31": GATES,
        "pass_count": pass_count,
        "pass_percent": pass_percent,
        "acceptance_passed": bool(acceptance_passed),
        "limitations": [
            "The 0.85 effective wetted-area utilization is a design requirement requiring CAD/CFD/experimental validation.",
            "The boiling check is a saturation-temperature margin, not a CHF correlation.",
            "No conjugate CFD, neutron transport, irradiation degradation, structural mechanics, or experimental validation.",
            "The 482-975 kg/s/module passing mass-flow range remains a plant-integration constraint.",
        ],
    }
    paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    zip_path = OUT_DIR / "Tomahawk_Cassette_Rev3_2_Area_Corrected_Results.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in paths.values():
            zf.write(path, arcname=path.name)

    print(summary.to_string(index=False))
    print("\nGate rates:\n", gate_summary.round(3).to_string(index=False))
    print("\nFailure matrix:\n", failures.to_string(index=False))
    print("\nSensitivity:\n", sensitivity.round(4).to_string(index=False))
    print("\nACCEPTANCE CHECK:", "PASS" if acceptance_passed else "FAIL")
    print("Results ZIP:", zip_path)
    return summary


if __name__ == "__main__":
    main()


Python: 3.12.13
Selected input: /kaggle/input/datasets/nicolebryant29/cassette-corrected-input/Tomahawk_Cassette_Rev3_2_Broad_Full_Incident_Area_Corrected_Input.csv
Rows: 10935
Input SHA-256: 027afd5d2cc160bfc769c6b3ba3ab45a611fc5163b90e21016bd5a88c55e14e5
Input validation: PASS
                                   analysis_branch  rows  legacy_rev31_passing_rows  legacy_rev31_pass_percent  rev32_passing_rows  rev32_pass_percent      acceptance_rule  acceptance_passed  elapsed_seconds  minimum_passing_mass_flow_kg_s_module  maximum_passing_mass_flow_kg_s_module
full_incident_broad_challenge_rev32_area_corrected 10935                        672                   6.145405                6974           63.776863 pass_percent >= 60.0               True          0.01708                             482.057416                             974.880383

Gate rates:
                gate  passing_rows  pass_percent
   gate_temperature         10839        99.122
gate_boiling_margin         10211     